# Scopus and OpenAlex Bronze-to-Silver evidence

This notebook reads the same mounted Bronze and Silver directories used by the scheduled container. Run it after an ingestion/processing cycle to inspect a live snapshot. It is intentionally read-only: it does not call either API and never accesses credentials.

## Evidence checklist

1. Batch metadata per source and status (ingestion checkpoints).
2. Raw Bronze Parquet pages per source.
3. Scopus window planning: oversized windows are split until each one fits the 5,000-result limit, so early cycles can hold many windows with zero records.
4. Per-batch Spark processing results.
5. Silver (schema v3) rows, quality flags and DOI-deduplicated analysis view.
6. Rejected rows with their reasons, including compacted groups.

Empty zones are reported clearly, so the notebook is also safe to run before the first scheduled batch.

In [ ]:
from pathlib import Path
import json
import sys

from pyspark.sql import SparkSession, functions as F

BRONZE = Path("/data/bronze")
SILVER = Path("/data/silver")
PROJECT_ROOT = Path("/app")
if PROJECT_ROOT.exists() and str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
spark = (
    SparkSession.builder.master("local[*]")
    .appName("openalex-pipeline-evidence")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# 1. Batch metadata is the audit index for ingestion and processing state.
metadata_rows = []
for path in sorted((BRONZE / "batch_metadata").glob("*.json")):
    item = json.loads(path.read_text(encoding="utf-8"))
    metadata_rows.append({
        "source": str(item.get("source", "openalex")),
        "batch_id": str(item.get("batch_id")),
        "mode": str(item.get("mode") or "unknown"),
        "status": str(item.get("status") or "unknown"),
        "stop_reason": str(item.get("stop_reason") or ""),
        "record_count": int(item.get("record_count", 0) or 0),
        "page_count": int(item.get("page_count", 0) or 0),
        "total_results": int(item.get("total_results", 0) or 0),
        "created_at": str(item.get("created_at") or "unknown"),
    })
if metadata_rows:
    batch_metadata_df = spark.createDataFrame(metadata_rows)
    print(f"Batch metadata rows: {batch_metadata_df.count()}")
    (
        batch_metadata_df.groupBy("source", "status")
        .agg(F.count("*").alias("batches"), F.sum("record_count").alias("records"), F.sum("page_count").alias("pages"))
        .orderBy("source", "status")
        .show(truncate=False)
    )
else:
    print("No batch metadata has been written yet.")

In [ ]:
# 2. Raw pages are Parquet files; count them independently from auxiliary Spark files.
raw_page_counts = [
    {"source": source, "raw_page_files": len(list((BRONZE / source).glob("*/page-*.parquet")))}
    for source in ("openalex", "scopus")
]
spark.createDataFrame(raw_page_counts).orderBy("source").show(truncate=False)
if not any(row["raw_page_files"] for row in raw_page_counts):
    print("No raw Bronze pages have been written yet.")

In [ ]:
# 3. Scopus windows: 'oversized' means the window was split further, 'completed_exhausted'
# means it was fully read, and 'time_budget_reached' means the cycle ended mid-plan.
scopus_df = batch_metadata_df.filter(F.col("source") == "scopus") if metadata_rows else None
if scopus_df is not None and scopus_df.count():
    scopus_df.groupBy("status", "stop_reason").agg(
        F.count("*").alias("windows"), F.sum("record_count").alias("records")
    ).orderBy(F.desc("windows")).show(truncate=False)
    oversized = scopus_df.filter(F.col("status") == "oversized")
    print(f"Oversized windows already split: {oversized.count()}")
    scopus_df.filter(F.col("record_count") > 0).select(
        "batch_id", "status", "record_count", "total_results"
    ).orderBy("created_at").show(20, truncate=False)
else:
    print("No Scopus windows have been planned yet.")

In [ ]:
# 4. Processing state records valid/rejected rows and write duration per batch.
processing_rows = []
for item in metadata_rows:
    if item["page_count"] == 0:
        continue  # empty windows are never processed
    state_path = SILVER / "_processing" / f"{item['batch_id']}.json"
    state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {}
    processing_rows.append({
        "source": item["source"],
        "batch_id": item["batch_id"],
        "ingestion_status": item["status"],
        "processing_status": str(state.get("status", "not_processed")),
        "valid_rows": int(state.get("valid_rows", -1)),
        "rejected_rows": int(state.get("rejected_rows", -1)),
        "write_seconds": float(state.get("write_seconds", -1.0)),
    })
if processing_rows:
    processing_df = spark.createDataFrame(processing_rows)
    done = sum(row["processing_status"] == "completed" for row in processing_rows)
    print(f"Batches with data: {len(processing_rows)}; processed: {done}  (-1 means not available yet)")
    processing_df.orderBy("source", "batch_id").show(50, truncate=False)
else:
    print("No ingestion batches with data are available to process yet.")

In [ ]:
# 5. Silver reads only the published version of each batch group (compaction manifest).
from src.compaction import active_silver_paths
from src.spark_pipeline import unique_publications

silver_paths = active_silver_paths(SILVER)
silver_df = None
if silver_paths:
    silver_df = spark.read.option("mergeSchema", "true").parquet(*(str(path) for path in silver_paths))
    print(f"Silver rows read: {silver_df.count():,}")
    silver_df.printSchema()
    silver_df.groupBy("source").count().orderBy("source").show()
    silver_df.select(
        "source", "source_id", "batch_id", "title", "publication_year", "normalized_doi"
    ).orderBy("source", "source_id").show(10, truncate=60)
    unique_df = unique_publications(silver_df)
    print(f"DOI-deduplicated analysis-view rows: {unique_df.count():,}")
    unique_df.groupBy("source").count().orderBy("source").show()
else:
    print("No published Silver Parquet batches have been written yet.")

In [ ]:
# 6. Schema v3 profile: coverage of the NLP, topic, authorship and geography columns.
# Scopus standard view has no abstract or topics, so low coverage there is expected.
if silver_df is not None:
    silver_df.groupBy("source").agg(
        F.count("*").alias("rows"),
        F.round(F.avg(F.col("has_abstract").cast("int")), 3).alias("has_abstract"),
        F.round(F.avg(F.col("has_topics").cast("int")), 3).alias("has_topics"),
        F.round(F.avg(F.col("author_count")), 2).alias("avg_authors"),
        F.round(F.avg((F.size("country_codes") > 0).cast("int")), 3).alias("has_country"),
        F.round(F.avg(F.col("cited_by_count")), 2).alias("avg_citations"),
        F.min("publication_year").alias("min_year"),
        F.max("publication_year").alias("max_year"),
    ).orderBy("source").show(truncate=False)
    silver_df.groupBy("publication_year").pivot("source").count().orderBy("publication_year").show(30)
    silver_df.select("source", "title", "primary_topic", "topics", "authors", "country_codes").where(
        F.col("has_topics")
    ).limit(3).show(truncate=80, vertical=True)
else:
    print("Silver profile skipped: no Silver data yet.")

In [ ]:
# 7. Rejection reasons, reading compacted groups plus batches not yet compacted.
from src.compaction import active_rejected_paths

rejected_paths = [path for path in active_rejected_paths(SILVER) if path.exists()]
if rejected_paths:
    rejected_df = spark.read.parquet(*(str(path) for path in rejected_paths))
    print(f"Rejected rows: {rejected_df.count()}")
    rejected_df.groupBy("rejection_reason").count().orderBy(F.desc("count")).show(truncate=False)
else:
    print("No rejected rows have been written yet.")